# Регрессия IC50 с подбором гиперпараметров

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [3]:
# Загрузка данных
df = pd.read_excel('../data/Данные_для_курсовои_Классическое_МО.xlsx')
df = df.drop(columns=['Unnamed: 0'])
df = df.dropna()

In [4]:
# Подготовка данных
X = df.drop(columns=['IC50, mM', 'CC50, mM', 'SI'])
y = np.log1p(df['IC50, mM'])                          # логарифмируем из-за скошенности

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [5]:
# Удаляем коррелированные признаки с порогом 0.95 (только на train)
corr = X_train.corr()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(abs(upper[col]) > 0.95)]

# Применяем удаление и к train, и к test
X_train = X_train.drop(columns=to_drop)
X_test = X_test.drop(columns=to_drop)

print(f"Удалено признаков: {len(to_drop)}")
print(f"Признаков в train: {X_train.shape[1]}")
print(f"Признаков в test: {X_test.shape[1]}")

Удалено признаков: 33
Признаков в train: 177
Признаков в test: 177


Применим следующие модели:  
- "LinearRegression"
- "RandomForest"
- "GradientBoosting"
- "CatBoost"
- "XGBoost"

**Подбор гиперпараметров не проводил из-за высокой вычислительной стоимости: полный перебор занимал более 3 часов.**

In [ ]:
# пример
'''
пример:  
models_ic50 = {  
    'XGBoost': {  
        'model': XGBRegressor(random_state=42, n_jobs=-1),  
        'params': {  
            'n_estimators': [100, 200, 300, 400],  
            'max_depth': [3, 5, 7, 9],  
            'learning_rate': [0.01, 0.03, 0.05, 0.1],  
            'subsample': [0.7, 0.8, 0.9, 1.0],  
            'colsample_bytree': [0.7, 0.8, 0.9, 1.0]  
        }  
    },  
    'CatBoost': {  
        'model': CatBoostRegressor(random_seed=42, verbose=0),  
        'params': {
            'iterations': [300, 500, 700],
            'depth': [4, 6, 8, 10],
            'learning_rate': [0.01, 0.03, 0.05, 0.1],
            'l2_leaf_reg': [1, 3, 5, 7]
        }
    },
    'RandomForest': {
        'model': RandomForestRegressor(random_state=42, n_jobs=-1),
        'params': {
            'n_estimators': [200, 300, 400, 500],
            'max_depth': [10, 15, 20, None],
            'min_samples_split': [2, 5, 10, 15],
            'min_samples_leaf': [1, 2, 4, 6]
        }
    }
}
'''

In [6]:
# ВСЕ 5 МОДЕЛЕЙ
models = {
    "LinearRegression": Pipeline([        # Линейная регрессия с масштабированием
        ('scaler', RobustScaler()),       # Логистическая регрессия чувствительна к масштабу
        ('model', LinearRegression())
    ]),
    "RandomForest": RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1),
    "GradientBoosting": GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42),
    "CatBoost": CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6, verbose=0, random_seed=42),
    "XGBoost": XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=5, random_state=42, n_jobs=-1)
}

# Обучение и сравнение
results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred_log = model.predict(X_test)
    
    y_pred = np.expm1(y_pred_log)
    y_true = np.expm1(y_test)
    
    results.append({
        "model": name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred)
    })

results_df = pd.DataFrame(results).sort_values("R2", ascending=False)

print("\n" + "="*48)
print("РЕЗУЛЬТАТЫ СРАВНЕНИЯ ВСЕХ 5 МОДЕЛЕЙ:")
print("="*48)
print(results_df.to_string(index=False))


РЕЗУЛЬТАТЫ СРАВНЕНИЯ ВСЕХ 5 МОДЕЛЕЙ:
           model        MAE       RMSE       R2
        CatBoost 183.087083 387.526437 0.456324
         XGBoost 194.282390 423.202015 0.351615
GradientBoosting 199.932927 424.046164 0.349026
    RandomForest 193.724409 429.516061 0.332123
LinearRegression 216.752192 503.814726 0.081077


**Итог**

Лучшая модель — CatBoost  

MAE (ошибка): самая маленькая → 183.09  
RMSE: самая маленькая → 387.53  
R²: самый высокий → 0.456  

То есть:  
ошибки минимальные  
модель лучше всего объясняет данные (~45.6% дисперсии)  

**Вывод**

По результатам сравнительного анализа моделей наилучшее качество продемонстрировала модель CatBoost, показав минимальные значения метрик MAE и RMSE, а также наибольшее значение коэффициента детерминации (R² = 0.456). Это свидетельствует о том, что данная модель наиболее точно предсказывает целевую переменную и лучше других улавливает зависимости в данных.  

Модели XGBoost, Gradient Boosting и Random Forest продемонстрировали сопоставимые, но несколько более слабые результаты. Линейная регрессия показала наихудшее качество, что указывает на нелинейный характер зависимости между признаками и целевой переменной.  

В целом полученные результаты можно оценить как удовлетворительные: модель CatBoost объясняет около 45.6% вариации целевой переменной, однако остаётся потенциал для повышения качества за счёт дальнейшей настройки моделей, отбора признаков и расширения набора данных.  